In [1]:
import os

In [31]:
%pwd

'/Users/User/Documents/mlops'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/Users/User/Documents/mlops'

In [11]:
import pandas as pd

df_train = pd.read_csv("/Users/User/Documents/mlops/artifacts/data_ingestion/Final_train_dataset.csv")
df_test = pd.read_csv("/Users/User/Documents/mlops/artifacts/data_ingestion/Final_test_dataset.csv")

In [12]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 249 entries, 0 to 248
Data columns (total 57 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   RecordID                249 non-null    object 
 1   Season                  249 non-null    object 
 2   Team                    249 non-null    object 
 3   Conference              249 non-null    object 
 4   Overall Seed            249 non-null    float64
 5   Bid Type                249 non-null    object 
 6   NET Rank                249 non-null    float64
 7   PrevNET                 249 non-null    float64
 8   AvgOppNETRank           249 non-null    float64
 9   AvgOppNET               249 non-null    float64
 10  WL                      249 non-null    object 
 11  Conf.Record             249 non-null    object 
 12  Non-ConferenceRecord    249 non-null    object 
 13  RoadWL                  249 non-null    object 
 14  NETSOS                  249 non-null    fl

In [13]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 451 entries, 0 to 450
Data columns (total 56 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   RecordID                451 non-null    object 
 1   Season                  451 non-null    object 
 2   Team                    451 non-null    object 
 3   Conference              451 non-null    object 
 4   Bid Type                91 non-null     object 
 5   NET Rank                446 non-null    float64
 6   PrevNET                 446 non-null    float64
 7   AvgOppNETRank           446 non-null    float64
 8   AvgOppNET               446 non-null    float64
 9   WL                      451 non-null    object 
 10  Conf.Record             451 non-null    object 
 11  Non-ConferenceRecord    451 non-null    object 
 12  RoadWL                  451 non-null    object 
 13  NETSOS                  446 non-null    float64
 14  NETNonConfSOS           439 non-null    fl

In [14]:
df_train.isna().sum()

RecordID                  0
Season                    0
Team                      0
Conference                0
Overall Seed              0
Bid Type                  0
NET Rank                  0
PrevNET                   0
AvgOppNETRank             0
AvgOppNET                 0
WL                        0
Conf.Record               0
Non-ConferenceRecord      0
RoadWL                    0
NETSOS                    0
NETNonConfSOS             1
Quadrant1                 0
Quadrant2                 0
Quadrant3                 0
Quadrant4                 0
Wins                      0
Losses                    0
WinsPct                   0
ConfWins                  0
ConfLosses                0
ConfWinsPct               0
NonConfWins               0
NonConfLosses             0
NonConfWinsPct            0
RoadWins                  0
RoadLosses                0
RoadWinsPct               0
Q1Wins                    0
Q1Losses                  0
Q1WinsPct                 0
Q2Wins              

In [15]:
df_test.isna().sum()

RecordID                    0
Season                      0
Team                        0
Conference                  0
Bid Type                  360
NET Rank                    5
PrevNET                     5
AvgOppNETRank               5
AvgOppNET                   5
WL                          0
Conf.Record                 0
Non-ConferenceRecord        0
RoadWL                      0
NETSOS                      5
NETNonConfSOS              12
Quadrant1                   0
Quadrant2                   0
Quadrant3                   0
Quadrant4                   0
Wins                        0
Losses                      0
WinsPct                     0
ConfWins                    0
ConfLosses                  0
ConfWinsPct                 0
NonConfWins                 0
NonConfLosses               0
NonConfWinsPct              0
RoadWins                    0
RoadLosses                  0
RoadWinsPct                 0
Q1Wins                      0
Q1Losses                    0
Q1WinsPct 

In [40]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataValidationConfig:
    root_dir: Path
    STATUS_FILE: str
    unzip_train_data_dir: Path
    unzip_test_data_dir: Path
    all_schema: dict

In [41]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation
        train_schema = self.schema.TRAIN_COLUMNS
        test_schema = self.schema.TEST_COLUMNS

        create_directories([config.root_dir])

        data_validation_config = DataValidationConfig(
            root_dir=config.root_dir,  # Need to fix this part
            STATUS_FILE= config.STATUS_FILE,
            unzip_train_data_dir = config.unzip_train_data_dir,
            unzip_test_data_dir = config.unzip_test_data_dir,
            all_schema = {"TRAIN_COLUMNS": train_schema, "TEST_COLUMNS": test_schema},
        )

        return data_validation_config

In [46]:
import os
from mlProject import logger

class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config


    def validate_all_columns(self)-> bool:
        try:
            validation_status = None

            for data_path, schema_key in [
                (self.config.unzip_train_data_dir, "TRAIN_COLUMNS"),
                (self.config.unzip_test_data_dir, "TEST_COLUMNS")
            ]:
                data = pd.read_csv(data_path)
                expected_cols = self.config.all_schema[schema_key].keys()

                for col in data.columns:
                    if col not in expected_cols:
                        validation_status = False
                        with open(self.config.STATUS_FILE, 'w') as f:
                            f.write(f"Validation status: {validation_status}")
                else:
                    validation_status = True
                    with open(self.config.STATUS_FILE, 'w') as f:
                        f.write(f"Validation status: {validation_status}")

            return validation_status
        
        except Exception as e:
            raise e
        


In [47]:
try:
    config = ConfigurationManager()
    data_validation_config = config.get_data_validation_config()
    data_validation = DataValidation(config=data_validation_config)
    data_validation.validate_all_columns()
except Exception as e:
    raise e

[2026-03-05 19:17:26,738: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-03-05 19:17:26,741: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-05 19:17:26,751: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-03-05 19:17:26,753: INFO: common: created directory at: artifacts]
[2026-03-05 19:17:26,753: INFO: common: created directory at: artifacts/data_validation]
